In [1]:
import os

In [2]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/ntchindagiscard/Documents/end-end-mlflow'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataProcessingConfig:
    root_dir: Path
    movies: Path
    ratings: Path
    tags: Path

In [6]:
from mlProject.utils.common import read_yaml, create_directories
from mlProject.constants import *

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH
            ) -> None:
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_processing_config(self) -> DataProcessingConfig:

        config  = self.config.data_processing

        create_directories([config.root_dir])

        data_processing_config = DataProcessingConfig(
            movies= config.movies,
            tags= config.tags,
            ratings= config.ratings,
            root_dir= config.root_dir
        )

        return data_processing_config

In [107]:
import pandas as pd

config = ConfigurationManager()
data_processing_config = config.get_data_processing_config()

movies_df = pd.read_csv(data_processing_config.movies)
ratings_df = pd.read_csv(data_processing_config.ratings)
tags_df = pd.read_csv(data_processing_config.tags)
print(data_processing_config)

[2025-01-18 09:29:14,001: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2025-01-18 09:29:14,154: INFO: common: Yaml file : params.yaml loaded successfully]
[2025-01-18 09:29:14,210: INFO: common: Yaml file : schema.yaml loaded successfully]
[2025-01-18 09:29:14,263: INFO: common: Created directory at: artifacts]
[2025-01-18 09:29:14,327: INFO: common: Created directory at: artifacts/data_transformation]


DataProcessingConfig(root_dir='artifacts/data_transformation', movies='artifacts/data_ingestion/ml-latest-small/movies.csv', ratings='artifacts/data_ingestion/ml-latest-small/ratings.csv', tags='artifacts/data_ingestion/ml-latest-small/tags.csv')


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Load datasets
movies_df = pd.read_csv(data_processing_config.movies)
ratings_df = pd.read_csv(data_processing_config.ratings)
tags_df = pd.read_csv(data_processing_config.tags)

# Select 5 users who have rated the most movies
top_5_users = ratings_df['userId'].value_counts().head(50).index
ratings_df = ratings_df[ratings_df['userId'].isin(top_5_users)]

# Filter movies and tags to only include those related to these users
relevant_movies = ratings_df['movieId'].unique()
movies_df = movies_df[movies_df['movieId'].isin(relevant_movies)]
tags_df = tags_df[tags_df['userId'].isin(top_5_users)]

def calculate_user_genre_ratings(ratings_df, movies_df):
    # Create genre columns
    genres = movies_df['genres'].str.get_dummies('|')
    movies_with_genres = pd.concat([movies_df[['movieId']], genres], axis=1)
    
    # Merge ratings with movies and genres
    ratings_with_genres = ratings_df.merge(movies_with_genres, on='movieId')
    
    # Calculate average rating per genre per user
    genre_columns = genres.columns
    user_genre_ratings = []
    
    for user_id in ratings_with_genres['userId'].unique():
        user_ratings = ratings_with_genres[ratings_with_genres['userId'] == user_id]
        genre_avgs = {}
        genre_avgs['userId'] = user_id
        
        for genre in genre_columns:
            genre_movies = user_ratings[user_ratings[genre] == 1]
            genre_avgs[f'{genre}_avg_rating'] = genre_movies['rating'].mean() if len(genre_movies) > 0 else 0
            
        user_genre_ratings.append(genre_avgs)
    
    return pd.DataFrame(user_genre_ratings)

def prepare_user_features(ratings_df, movies_df, tags_df):
    # Basic user statistics
    rating_stats = ratings_df.groupby('userId').agg({
        'rating': ['mean', 'std', 'count']
    }).fillna(0)
    rating_stats.columns = ['avg_rating', 'std_rating', 'rating_count']
    rating_stats = rating_stats.reset_index()
    
    # Tag statistics
    tag_stats = tags_df.groupby('userId').agg({
        'tag': 'count'
    }).reset_index()
    tag_stats.columns = ['userId', 'tag_count']
    
    # Genre rating averages
    genre_ratings = calculate_user_genre_ratings(ratings_df, movies_df)
    
    # Combine all user features
    user_features = rating_stats.merge(tag_stats, on='userId', how='left')
    user_features = user_features.merge(genre_ratings, on='userId', how='left')
    user_features = user_features.fillna(0)
    
    # Save user IDs before normalization
    user_ids = user_features['userId']
    
    # Standardize all features
    feature_columns = [col for col in user_features.columns if col != 'userId']
    user_features_normalized = StandardScaler().fit_transform(user_features[feature_columns])
    
    return user_features_normalized, user_features

def prepare_movie_features(movies_df, tags_df):
    # Genre features
    genres = movies_df['genres'].str.get_dummies('|')
    
    # Extract year
    movies_df['year'] = movies_df['title'].str.extract('(\d{4})', expand=False)
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
    year_normalized = StandardScaler().fit_transform(movies_df[['year']].fillna(movies_df['year'].mean()))
    
    # Tag features
    movie_tags = tags_df.groupby('movieId')['tag'].agg(lambda x: ' '.join(x)).reset_index()
    movie_tags = movies_df[['movieId']].merge(movie_tags, on='movieId', how='left')
    movie_tags['tag'] = movie_tags['tag'].fillna('')
    
    # Reduce TF-IDF features for the sample
    tfidf = TfidfVectorizer(max_features=50, stop_words='english')
    tag_features = tfidf.fit_transform(movie_tags['tag']).toarray()
    
    # Combine all features
    movie_features = np.hstack([
        genres.values,
        year_normalized,
        tag_features
    ])
    
    return movie_features, tfidf

def prepare_training_data(ratings_df, movie_features, user_features):
    user_encoder = LabelEncoder()
    movie_encoder = LabelEncoder()
    
    ratings_df['user_encoded'] = user_encoder.fit_transform(ratings_df['userId'])
    ratings_df['movie_encoded'] = movie_encoder.fit_transform(ratings_df['movieId'])
    
    X_user = user_features[ratings_df['user_encoded']]
    X_movie = movie_features[ratings_df['movie_encoded']]
    y = ratings_df['rating'].values
    
    return X_user, X_movie, y, user_encoder, movie_encoder

# Execute the pipeline
movie_features, tfidf = prepare_movie_features(movies_df, tags_df)
user_features, user_features_df = prepare_user_features(ratings_df, movies_df, tags_df)
X_user, X_movie, y, user_encoder, movie_encoder = prepare_training_data(ratings_df, movie_features, user_features)


# Display information about the processed data
print("Dataset Overview:")
print(f"Number of users: {len(top_5_users)}")
print(f"Number of movies: {len(movies_df)}")
print(f"Number of ratings: {len(ratings_df)}")

print("\nSample of User Features:")
print("\nUser Statistics:")
print(user_features_df[['userId', 'avg_rating', 'std_rating', 'rating_count', 'tag_count']].to_string())

print("\nSample of Genre Preferences for first user:")
genre_columns = [col for col in user_features_df.columns if '_avg_rating' in col]
print(user_features_df[['userId'] + genre_columns].iloc[20].to_string())

print("\nFeature Dimensions:")
print(f"User features shape: {user_features.shape}")
print(f"Movie features shape: {movie_features.shape}")

# Split data
train_size = 0.8
indices = np.arange(len(y))
train_indices, val_indices = train_test_split(indices, train_size=train_size, random_state=42)

# Create final sets
train_user_features = X_user[train_indices]
train_movie_features = X_movie[train_indices]
train_ratings = y[train_indices]

val_user_features = X_user[val_indices]
val_movie_features = X_movie[val_indices]
val_ratings = y[val_indices]

print("\nTraining/Validation Split:")
print(f"Training samples: {len(train_indices)}")
print(f"Validation samples: {len(val_indices)}")

## OOP


In [8]:
# base feature transformer
from abc import ABC, abstractmethod
import pandas as pd
from typing import Optional

class FeatureExtractor(ABC):

    @abstractmethod
    def generate_features(self, ratings_df: Optional[pd.DataFrame], movies_df: Optional[pd.DataFrame], tags_df: Optional[pd.DataFrame] ) -> pd.DataFrame:
        pass

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path
import numpy as np

class MovieFeatureExtractor(FeatureExtractor):   

    def generate_features(self, movies_df : pd.DataFrame, tags_df: pd.DataFrame) -> np.ndarray:
        genres = movies_df['genres'].str.get_dummies('|')
    
        # Extract year
        movies_df['year'] = movies_df['title'].str.extract('(\d{4})', expand=False)
        movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
        year_normalized = StandardScaler().fit_transform(movies_df[['year']].fillna(movies_df['year'].mean()))
        
        # Tag features
        movie_tags = tags_df.groupby('movieId')['tag'].agg(lambda x: ' '.join(x)).reset_index()
        movie_tags = movies_df[['movieId']].merge(movie_tags, on='movieId', how='left')
        movie_tags['tag'] = movie_tags['tag'].fillna('')
        
        # Reduce TF-IDF features for the sample
        tfidf = TfidfVectorizer(max_features=50, stop_words='english')
        tag_features = tfidf.fit_transform(movie_tags['tag']).toarray()
        
        # Combine all features
        movie_features = np.hstack([
            genres.values,
            year_normalized,
            tag_features
        ])
        
        return movie_features

In [25]:
class UserFeatureExtractor(FeatureExtractor):
    """
    Extracts user features from ratings and movies data.

    Attributes:
        None

    Methods:
        generate_features: Generates user features based on ratings and movies data.

    """

    def calculate_user_genre_ratings(self, ratings_df, movies_df):
        # Create genre columns
        genres = movies_df['genres'].str.get_dummies('|')
        movies_with_genres = pd.concat([movies_df[['movieId']], genres], axis=1)
        
        # Merge ratings with movies and genres
        ratings_with_genres = ratings_df.merge(movies_with_genres, on='movieId')
        
        # Calculate average rating per genre per user
        genre_columns = genres.columns
        user_genre_ratings = []
        
        for user_id in ratings_with_genres['userId'].unique():
            user_ratings = ratings_with_genres[ratings_with_genres['userId'] == user_id]
            genre_avgs = {}
            genre_avgs['userId'] = user_id
            
            for genre in genre_columns:
                genre_movies = user_ratings[user_ratings[genre] == 1]
                genre_avgs[f'{genre}_avg_rating'] = genre_movies['rating'].mean() if len(genre_movies) > 0 else 0
                
            user_genre_ratings.append(genre_avgs)
        
        return pd.DataFrame(user_genre_ratings)

    def generate_features(self, ratings_df: pd.DataFrame, movies_df: pd.DataFrame, tags_df: pd.DataFrame) -> pd.DataFrame:
        # Basic user statistics
        rating_stats = ratings_df.groupby('userId').agg({
            'rating': ['mean', 'std', 'count']
        }).fillna(0)
        rating_stats.columns = ['avg_rating', 'std_rating', 'rating_count']
        rating_stats = rating_stats.reset_index()
        
        # Tag statistics
        tag_stats = tags_df.groupby('userId').agg({
            'tag': 'count'
        }).reset_index()
        tag_stats.columns = ['userId', 'tag_count']
        
        # Genre rating averages
        genre_ratings = self.calculate_user_genre_ratings(ratings_df, movies_df)
        
        # Combine all user features
        user_features = rating_stats.merge(tag_stats, on='userId', how='left')
        user_features = user_features.merge(genre_ratings, on='userId', how='left')
        user_features = user_features.fillna(0)
        
        # Save user IDs before normalization
        user_ids = user_features['userId']
        
        # Standardize all features
        feature_columns = [col for col in user_features.columns if col != 'userId']
        user_features_normalized = StandardScaler().fit_transform(user_features[feature_columns])
        
        return user_features_normalized, user_features

In [36]:
from typing import Tuple
from mlProject import logger
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

class CSVDataPreprocessor:
    def __init__(self, config: DataProcessingConfig) -> None:
        self.movie_feature_extractor = MovieFeatureExtractor()
        self.user_feature_extractor = UserFeatureExtractor()
        self.config = config
        self.movies_df = pd.read_csv(self.config.movies)
        self.ratings_df = pd.read_csv(self.config.ratings)
        self.tags_df = pd.read_csv(self.config.tags)

        # Select 5 users who have rated the most movies
        self.top_5_users = self.ratings_df['userId'].value_counts().head(50).index
        self.ratings_df = self.ratings_df[self.ratings_df['userId'].isin(self.top_5_users)]

        # Filter movies and tags to only include those related to these users
        self.relevant_movies = self.ratings_df['movieId'].unique()
        self.movies_df = self.movies_df[self.movies_df['movieId'].isin(self.relevant_movies)]
        self.tags_df = self.tags_df[self.tags_df['userId'].isin(self.top_5_users)]
    
    def prepare_training_data(self, ratings_df, movie_features, user_features):
        user_encoder = LabelEncoder()
        movie_encoder = LabelEncoder()
        
        ratings_df['user_encoded'] = user_encoder.fit_transform(self.ratings_df['userId'])
        ratings_df['movie_encoded'] = movie_encoder.fit_transform(self.ratings_df['movieId'])
        
        X_user = user_features[ratings_df['user_encoded']]
        X_movie = movie_features[ratings_df['movie_encoded']]
        y = ratings_df['rating'].values
        
        return X_user, X_movie, y, user_encoder, movie_encoder
    
    def process_data(self) -> pd.DataFrame:
        
        logger.info(f"Extracting Movies features...⏳")
        movie_features = self.movie_feature_extractor.generate_features(self.movies_df, self.tags_df)
        logger.info(f"Extracting Movies features completed ✅ ")
        logger.info(f"Extracting User features...⏳")
        user_features, user_features_df = self.user_feature_extractor.generate_features(self.ratings_df, self.movies_df, self.tags_df)
        logger.info(f"Extracting User features completed ✅")
        logger.info(f"Merging features with ratings...⏳")
        X_user, X_movie, y, _, _ = self.prepare_training_data(self.ratings_df, movie_features, user_features)
        logger.info(f"Merging features with ratings completed ✅")

        return X_user, X_movie, y

    def train_validation_split(self, X_user, X_movie, y) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        train_size = 0.8
        indices = np.arange(len(y))
        train_indices, val_indices = train_test_split(indices, train_size=train_size, random_state=42)

        # Create final sets
        train_user_features = X_user[train_indices]
        train_movie_features = X_movie[train_indices]
        train_ratings = y[train_indices]

        val_user_features = X_user[val_indices]
        val_movie_features = X_movie[val_indices]
        val_ratings = y[val_indices]

        return train_user_features, train_movie_features, train_ratings, val_user_features, val_movie_features, val_ratings
    

In [37]:
try:
    config = ConfigurationManager()
    get_data_processing_config = config.get_data_processing_config()
    data_preprocessor = CSVDataPreprocessor(config=get_data_processing_config)
    x_user, x_movie, y = data_preprocessor.process_data()
    X_train_user, X_train_movie, y_train_rating, X_val_user, X_val_movie, y_val_rating = data_preprocessor.train_validation_split(x_user, x_movie, y)
except Exception as e:
    logger.exception(f"Oops😟! An error occured: {e} ")


[2021-01-01 03:34:31,470: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2021-01-01 03:34:31,510: INFO: common: Yaml file : params.yaml loaded successfully]
[2021-01-01 03:34:31,560: INFO: common: Yaml file : schema.yaml loaded successfully]
[2021-01-01 03:34:31,564: INFO: common: Created directory at: artifacts]
[2021-01-01 03:34:31,573: INFO: common: Created directory at: artifacts/data_transformation]


[2021-01-01 03:34:32,124: INFO: 3558107034: Extracting Movies features...⏳]
[2021-01-01 03:34:34,045: INFO: 3558107034: Extracting Movies features completed ✅ ]
[2021-01-01 03:34:34,049: INFO: 3558107034: Extracting User features...⏳]
[2021-01-01 03:34:38,805: INFO: 3558107034: Extracting User features completed ✅]
[2021-01-01 03:34:38,807: INFO: 3558107034: Merging features with ratings...⏳]
[2021-01-01 03:34:38,957: INFO: 3558107034: Merging features with ratings completed ✅]


# User Features


## Step 1: Aggregate Ratings


In [23]:
# Aggregate user ratings
user_ratings = ratings_df.groupby('userId')['rating'].agg(['mean', 'count']).reset_index()
user_ratings.rename(columns={'mean': 'avg_rating', 'count': 'rating_count'}, inplace=True)

user_ratings.head()

,userId,avg_rating,rating_count
0,1,4.366379,232
1,2,3.948276,29
2,3,2.435897,39
3,4,3.555556,216
4,5,3.636364,44


## Step 2: User Preference for Genres


In [24]:
user_genres = ratings_df.merge(movies_df[['movieId'] + list(genre_set)], on='movieId', how='left')
user_genres.head()

,userId,movieId,rating,timestamp,Western,Romance,Fantasy,Adventure,Action,Film-Noir,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,1,4.0,964982703,0,0,1,1,0,0,...,0,0,0,1,0,1,0,0,0,1
1,1,3,4.0,964981247,0,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2,1,6,4.0,964982224,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,1,47,5.0,964983815,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0
4,1,50,5.0,964982931,0,0,0,0,0,0,...,0,1,1,0,0,0,0,0,0,0


In [25]:
user_genre_prefenrences = user_genres.groupby('userId')[list(genre_set)].mean().reset_index()
user_genre_prefenrences.head()

,userId,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),IMAX,War,...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,0.030172,0.112069,0.202586,0.366379,0.387931,0.004310,0.0,0.000000,0.094828,...,0.172414,0.077586,0.237069,0.357759,0.073276,0.181034,0.094828,0.293103,0.000000,0.125000
1,2,0.034483,0.034483,0.000000,0.103448,0.379310,0.000000,0.0,0.137931,0.034483,...,0.137931,0.068966,0.344828,0.241379,0.034483,0.000000,0.000000,0.586207,0.103448,0.000000
2,3,0.000000,0.128205,0.102564,0.282051,0.358974,0.000000,0.0,0.000000,0.128205,...,0.384615,0.025641,0.179487,0.230769,0.205128,0.128205,0.025641,0.410256,0.000000,0.102564
3,4,0.046296,0.268519,0.087963,0.134259,0.115741,0.018519,0.0,0.004630,0.032407,...,0.055556,0.106481,0.175926,0.481481,0.018519,0.046296,0.074074,0.555556,0.009259,0.027778
4,5,0.045455,0.250000,0.159091,0.181818,0.204545,0.000000,0.0,0.068182,0.068182,...,0.045455,0.022727,0.204545,0.340909,0.022727,0.204545,0.113636,0.568182,0.000000,0.136364


In [26]:
user_features = user_ratings.merge(user_genre_prefenrences, on='userId', how='left')
print(user_features.count())
user_features.head()

userId                610
avg_rating            610
rating_count          610
Western               610
Romance               610
Fantasy               610
Adventure             610
Action                610
Film-Noir             610
(no genres listed)    610
IMAX                  610
War                   610
Crime                 610
Sci-Fi                610
Mystery               610
Thriller              610
Comedy                610
Horror                610
Children              610
Musical               610
Drama                 610
Documentary           610
Animation             610
dtype: int64


,userId,avg_rating,rating_count,Western,Romance,Fantasy,Adventure,Action,Film-Noir,(no genres listed),...,Sci-Fi,Mystery,Thriller,Comedy,Horror,Children,Musical,Drama,Documentary,Animation
0,1,4.366379,232,0.030172,0.112069,0.202586,0.366379,0.387931,0.004310,0.0,...,0.172414,0.077586,0.237069,0.357759,0.073276,0.181034,0.094828,0.293103,0.000000,0.125000
1,2,3.948276,29,0.034483,0.034483,0.000000,0.103448,0.379310,0.000000,0.0,...,0.137931,0.068966,0.344828,0.241379,0.034483,0.000000,0.000000,0.586207,0.103448,0.000000
2,3,2.435897,39,0.000000,0.128205,0.102564,0.282051,0.358974,0.000000,0.0,...,0.384615,0.025641,0.179487,0.230769,0.205128,0.128205,0.025641,0.410256,0.000000,0.102564
3,4,3.555556,216,0.046296,0.268519,0.087963,0.134259,0.115741,0.018519,0.0,...,0.055556,0.106481,0.175926,0.481481,0.018519,0.046296,0.074074,0.555556,0.009259,0.027778
4,5,3.636364,44,0.045455,0.250000,0.159091,0.181818,0.204545,0.000000,0.0,...,0.045455,0.022727,0.204545,0.340909,0.022727,0.204545,0.113636,0.568182,0.000000,0.136364


# implamentation


In [27]:
import os
from mlProject import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [28]:
class DataTransformation:

    def __init__(self, config: DataTransformationConfig) -> None:
        
        self.config = config

    def train_test_spliting(self):

        data = pd.read_csv(self.config.data_path)
        train,test = train_test_split(data)

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"), index=False)

        logger.info("Data splitted into test and training set")
        logger.info(train.shape)
        logger.info(test.shape)

In [29]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.train_test_spliting()

except Exception as e:
    raise e

[2021-01-01 12:55:06,840: INFO: common: Yaml file : config/config.yaml loaded successfully]
[2021-01-01 12:55:06,850: INFO: common: Yaml file : params.yaml loaded successfully]
[2021-01-01 12:55:06,867: INFO: common: Yaml file : schema.yaml loaded successfully]
[2021-01-01 12:55:06,874: INFO: common: Created directory at: artifacts]


AttributeError: 'ConfigurationManager' object has no attribute 'get_data_transformation_config'

In [ ]:
# scale training data
item_train_unscaled = item_train
user_train_unscaled = user_train
y_train_unscaled    = y_train

scalerItem = StandardScaler()
scalerItem.fit(item_train)
item_train = scalerItem.transform(item_train)

scalerUser = StandardScaler()
scalerUser.fit(user_train)
user_train = scalerUser.transform(user_train)

scalerTarget = MinMaxScaler((-1, 1))
scalerTarget.fit(y_train.reshape(-1, 1))
y_train = scalerTarget.transform(y_train.reshape(-1, 1))
#ynorm_test = scalerTarget.transform(y_test.reshape(-1, 1))

print(np.allclose(item_train_unscaled, scalerItem.inverse_transform(item_train)))
print(np.allclose(user_train_unscaled, scalerUser.inverse_transform(user_train)))

In [ ]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  
  
  
    ### END CODE HERE ###  
])

# create the user input and point to the base network
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# create the item input and point to the base network
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# compute the dot product of the two vectors vu and vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# specify the inputs and output of the model
model = tf.keras.Model([input_user, input_item], output)

model.summary()

In [ ]:
# model

from tensorflow.keras import layers, Model
import tensorflow as tf
import numpy as np
from typing import Tuple, List

class RecommenderNet(Model):
    def __init__(self, user_shape: int, movie_shape: int):
        super(RecommenderNet, self).__init__()
        
        # User tower layers
        self.user_input = layers.Input(shape=(user_shape,), name="user_input")
        self.user_dense1 = layers.Dense(64, activation="relu")
        self.user_dense2 = layers.Dense(32, activation="relu")
        
        # Movie tower layers
        self.movie_input = layers.Input(shape=(movie_shape,), name="movie_input")
        self.movie_dense1 = layers.Dense(64, activation="relu")
        self.movie_dense2 = layers.Dense(32, activation="relu")
        
        # Combined layers
        self.combined_dense = layers.Dense(64, activation="relu")
        self.output_layer = layers.Dense(1, activation="linear", name="output")
        
    def call(self, inputs):
        user_input, movie_input = inputs
        
        # User tower
        x1 = self.user_dense1(user_input)
        x1 = self.user_dense2(x1)
        
        # Movie tower
        x2 = self.movie_dense1(movie_input)
        x2 = self.movie_dense2(x2)
        
        # Combine towers
        combined = layers.concatenate([x1, x2])
        combined = self.combined_dense(combined)
        
        return self.output_layer(combined)
    
    def build_graph(self):
        """Create model graph for visualization"""
        model = Model(
            inputs=[self.user_input, self.movie_input],
            outputs=self.call([self.user_input, self.movie_input])
        )
        return model

class RecommenderTrainer:
    def __init__(
        self,
        user_shape: int,
        movie_shape: int,
        learning_rate: float = 0.001,
        batch_size: int = 32,
        epochs: int = 10
    ):
        self.model = RecommenderNet(user_shape, movie_shape)
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.epochs = epochs
        self.history = None
        
    def compile_model(self):
        """Compile the model with specified parameters"""
        optimizer = tf.keras.optimizers.Adam(learning_rate=self.learning_rate)
        self.model.compile(
            optimizer=optimizer,
            loss='mse',
            metrics=['mae']
        )
        
    def train(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray,
        validation_split: float = 0.2,
        callbacks: List = None
    ):
        """Train the model"""
        self.compile_model()
        self.history = self.model.fit(
            [X_user, X_movie],
            y,
            batch_size=self.batch_size,
            epochs=self.epochs,
            validation_split=validation_split,
            callbacks=callbacks
        )
        return self.history
    
    def predict(self, X_user: np.ndarray, X_movie: np.ndarray) -> np.ndarray:
        """Make predictions"""
        return self.model.predict([X_user, X_movie])
    
    def evaluate(
        self,
        X_user: np.ndarray,
        X_movie: np.ndarray,
        y: np.ndarray
    ) -> Tuple[float, float]:
        """Evaluate the model"""
        return self.model.evaluate([X_user, X_movie], y)
    
    def save_model(self, path: str):
        """Save the model"""
        self.model.save(path)
    
    @staticmethod
    def load_model(path: str):
        """Load a saved model"""
        return tf.keras.models.load_model(path)

# Usage example:
"""
# Initialize trainer
trainer = RecommenderTrainer(
    user_shape=X_user_np.shape[1],
    movie_shape=X_movie_np.shape[1],
    learning_rate=0.001,
    batch_size=32,
    epochs=10
)

# Define callbacks if needed
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
]

# Train the model
history = trainer.train(
    X_user_np,
    X_movie_np,
    y,
    validation_split=0.2,
    callbacks=callbacks
)

# Make predictions
predictions = trainer.predict(X_user_test, X_movie_test)

# Evaluate model
loss, mae = trainer.evaluate(X_user_test, X_movie_test, y_test)

# Save model
trainer.save_model('recommender_model.h5')

# Load model later
loaded_model = RecommenderTrainer.load_model('recommender_model.h5')
"""

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the datasets
ratings_df = pd.read_csv('ratings.csv')
movies_df = pd.read_csv('movies.csv')

# 1. Prepare Movie Features
def prepare_movie_features(movies_df):
    # Split genres into binary columns
    genres = movies_df['genres'].str.get_dummies('|')
    
    # Extract year from title
    movies_df['year'] = movies_df['title'].str.extract('(\d{4})', expand=False)
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')
    
    # Create title text features
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(movies_df['title'])
    title_sequences = tokenizer.texts_to_sequences(movies_df['title'])
    title_padded = pad_sequences(title_sequences, maxlen=10)
    
    # Combine features
    movie_features = np.hstack([
        genres.values,
        StandardScaler().fit_transform(movies_df[['year']].fillna(movies_df['year'].mean())),
        title_padded
    ])
    
    return movie_features, tokenizer

# 2. Prepare User Features
def prepare_user_features(ratings_df):
    # Calculate user statistics
    user_stats = ratings_df.groupby('userId').agg({
        'rating': ['mean', 'std', 'count']
    }).fillna(0)
    
    user_stats.columns = ['avg_rating', 'std_rating', 'rating_count']
    user_stats = user_stats.reset_index()
    
    # Normalize user features
    user_features = StandardScaler().fit_transform(user_stats[['avg_rating', 'std_rating', 'rating_count']])
    
    return user_features, user_stats

# 3. Prepare Training Data
def prepare_training_data(ratings_df, movie_features, user_features):
    # Create label encoders for user and movie IDs
    user_encoder = LabelEncoder()
    movie_encoder = LabelEncoder()
    
    ratings_df['user_encoded'] = user_encoder.fit_transform(ratings_df['userId'])
    ratings_df['movie_encoded'] = movie_encoder.fit_transform(ratings_df['movieId'])
    
    # Create training arrays
    X_user = user_features[ratings_df['user_encoded']]
    X_movie = movie_features[ratings_df['movie_encoded']]
    y = ratings_df['rating'].values
    
    return X_user, X_movie, y, user_encoder, movie_encoder

# Execute the preparation pipeline
movie_features, title_tokenizer = prepare_movie_features(movies_df)
user_features, user_stats = prepare_user_features(ratings_df)
X_user, X_movie, y, user_encoder, movie_encoder = prepare_training_data(ratings_df, movie_features, user_features)

# Split the data
train_size = 0.8
indices = np.arange(len(y))
train_indices, val_indices = train_test_split(indices, train_size=train_size, random_state=42)

# Create train and validation sets
train_user_features = X_user[train_indices]
train_movie_features = X_movie[train_indices]
train_ratings = y[train_indices]

val_user_features = X_user[val_indices]
val_movie_features = X_movie[val_indices]
val_ratings = y[val_indices]

# Print shapes to verify
print("Training set shapes:")
print(f"User features: {train_user_features.shape}")
print(f"Movie features: {train_movie_features.shape}")
print(f"Ratings: {train_ratings.shape}")

print("\nValidation set shapes:")
print(f"User features: {val_user_features.shape}")
print(f"Movie features: {val_movie_features.shape}")
print(f"Ratings: {val_ratings.shape}")

I'll break down the code into sections and explain each part in detail:

1. **Initial Imports and Data Loading**:

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load the datasets
ratings_df = pd.read_csv('ratings.csv')
movies_df = pd.read_csv('movies.csv')
```

- We import necessary libraries for data manipulation, preprocessing, and machine learning
- Load the MovieLens dataset files into pandas DataFrames

2. **Movie Features Preparation**:

```python
def prepare_movie_features(movies_df):
    # Split genres into binary columns
    genres = movies_df['genres'].str.get_dummies('|')

    # Extract year from title
    movies_df['year'] = movies_df['title'].str.extract('(\d{4})', expand=False)
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')

    # Create title text features
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(movies_df['title'])
    title_sequences = tokenizer.texts_to_sequences(movies_df['title'])
    title_padded = pad_sequences(title_sequences, maxlen=10)
```

This function processes movie features:

- **Genre Processing**: Converts the pipe-separated genres into binary columns (one-hot encoding)
  - For example, "Action|Adventure" becomes [1,1,0,0,...] where 1s represent presence of genres
- **Year Extraction**:
  - Extracts the year from movie titles using regex (\d{4})
  - Converts the extracted year to numeric format
- **Title Processing**:
  - Uses Keras Tokenizer to convert movie titles into numerical sequences
  - Limits vocabulary to 5000 most common words
  - Pads sequences to ensure uniform length (10 words)

3. **User Features Preparation**:

```python
def prepare_user_features(ratings_df):
    # Calculate user statistics
    user_stats = ratings_df.groupby('userId').agg({
        'rating': ['mean', 'std', 'count']
    }).fillna(0)

    user_stats.columns = ['avg_rating', 'std_rating', 'rating_count']
    user_stats = user_stats.reset_index()
```

This function creates user features:

- Groups ratings by user and calculates:
  - Mean rating (average rating given by user)
  - Standard deviation of ratings (rating variability)
  - Count of ratings (user activity level)
- Normalizes these features using StandardScaler
- These features help capture user rating behavior patterns

4. **Training Data Preparation**:

```python
def prepare_training_data(ratings_df, movie_features, user_features):
    # Create label encoders for user and movie IDs
    user_encoder = LabelEncoder()
    movie_encoder = LabelEncoder()

    ratings_df['user_encoded'] = user_encoder.fit_transform(ratings_df['userId'])
    ratings_df['movie_encoded'] = movie_encoder.fit_transform(ratings_df['movieId'])

    # Create training arrays
    X_user = user_features[ratings_df['user_encoded']]
    X_movie = movie_features[ratings_df['movie_encoded']]
    y = ratings_df['rating'].values
```

This function prepares the final training data:

- Encodes user and movie IDs into consecutive integers using LabelEncoder
- Creates three main components:
  - X_user: User feature matrix
  - X_movie: Movie feature matrix
  - y: Target ratings

5. **Data Split and Final Preparation**:

```python
# Split the data
train_size = 0.8
indices = np.arange(len(y))
train_indices, val_indices = train_test_split(indices, train_size=train_size, random_state=42)

# Create train and validation sets
train_user_features = X_user[train_indices]
train_movie_features = X_movie[train_indices]
train_ratings = y[train_indices]

val_user_features = X_user[val_indices]
val_movie_features = X_movie[val_indices]
val_ratings = y[val_indices]
```

Final steps:

- Splits data into training (80%) and validation (20%) sets
- Creates separate arrays for:
  - Training: user features, movie features, and ratings
  - Validation: user features, movie features, and ratings

The resulting data structure is ready for a two-tower neural network where:

- One tower processes user features
- Another tower processes movie features
- The network learns to predict ratings based on these features

The final shapes printed show:

- Number of training and validation examples
- Dimensionality of user and movie feature vectors
- Target rating vector shape

This preprocessed data can now be fed into a neural network model where:

- User tower will learn user embeddings from user features
- Movie tower will learn movie embeddings from movie features
- The two towers' outputs will be combined to predict ratings

Would you like me to provide the code for the neural network architecture that would use this preprocessed data?


Yes, absolutely! We can enhance our feature extraction by incorporating the tags.csv file. Tags provide additional semantic information about movies. Here's the modified code that includes tag processing:

```python
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load all datasets
ratings_df = pd.read_csv('ratings.csv')
movies_df = pd.read_csv('movies.csv')
tags_df = pd.read_csv('tags.csv')

def prepare_tag_features(movies_df, tags_df, max_features=1000):
    # Group tags by movieId and concatenate them
    movie_tags = tags_df.groupby('movieId')['tag'].agg(lambda x: ' '.join(x)).reset_index()

    # Merge with movies_df to ensure we have entries for all movies
    movie_tags = pd.merge(movies_df[['movieId']], movie_tags, on='movieId', how='left')
    movie_tags['tag'] = movie_tags['tag'].fillna('')

    # Create TF-IDF features from tags
    tfidf = TfidfVectorizer(max_features=max_features,
                           stop_words='english',
                           lowercase=True,
                           analyzer='word',
                           ngram_range=(1, 2))

    tag_features = tfidf.fit_transform(movie_tags['tag'])

    return tag_features.toarray(), tfidf

def prepare_movie_features(movies_df, tags_df):
    # Split genres into binary columns
    genres = movies_df['genres'].str.get_dummies('|')

    # Extract year from title
    movies_df['year'] = movies_df['title'].str.extract('(\d{4})', expand=False)
    movies_df['year'] = pd.to_numeric(movies_df['year'], errors='coerce')

    # Create title text features
    tokenizer = Tokenizer(num_words=5000)
    tokenizer.fit_on_texts(movies_df['title'])
    title_sequences = tokenizer.texts_to_sequences(movies_df['title'])
    title_padded = pad_sequences(title_sequences, maxlen=10)

    # Get tag features
    tag_features, tfidf = prepare_tag_features(movies_df, tags_df)

    # Combine all features
    movie_features = np.hstack([
        genres.values,
        StandardScaler().fit_transform(movies_df[['year']].fillna(movies_df['year'].mean())),
        title_padded,
        tag_features  # Add tag features to the movie features
    ])

    return movie_features, tokenizer, tfidf

def prepare_user_features(ratings_df, tags_df):
    # Calculate rating statistics
    rating_stats = ratings_df.groupby('userId').agg({
        'rating': ['mean', 'std', 'count']
    }).fillna(0)

    rating_stats.columns = ['avg_rating', 'std_rating', 'rating_count']

    # Calculate tagging activity
    tag_stats = tags_df.groupby('userId').agg({
        'tag': 'count'
    }).reset_index()
    tag_stats.columns = ['userId', 'tag_count']

    # Merge rating and tag statistics
    user_stats = rating_stats.reset_index().merge(
        tag_stats, on='userId', how='left'
    ).fillna(0)

    # Normalize user features
    user_features = StandardScaler().fit_transform(
        user_stats[['avg_rating', 'std_rating', 'rating_count', 'tag_count']]
    )

    return user_features, user_stats

def prepare_training_data(ratings_df, movie_features, user_features):
    # Create label encoders for user and movie IDs
    user_encoder = LabelEncoder()
    movie_encoder = LabelEncoder()

    ratings_df['user_encoded'] = user_encoder.fit_transform(ratings_df['userId'])
    ratings_df['movie_encoded'] = movie_encoder.fit_transform(ratings_df['movieId'])

    # Create training arrays
    X_user = user_features[ratings_df['user_encoded']]
    X_movie = movie_features[ratings_df['movie_encoded']]
    y = ratings_df['rating'].values

    return X_user, X_movie, y, user_encoder, movie_encoder

# Execute the preparation pipeline
movie_features, title_tokenizer, tfidf = prepare_movie_features(movies_df, tags_df)
user_features, user_stats = prepare_user_features(ratings_df, tags_df)
X_user, X_movie, y, user_encoder, movie_encoder = prepare_training_data(ratings_df, movie_features, user_features)

# Split the data
train_size = 0.8
indices = np.arange(len(y))
train_indices, val_indices = train_test_split(indices, train_size=train_size, random_state=42)

# Create train and validation sets
train_user_features = X_user[train_indices]
train_movie_features = X_movie[train_indices]
train_ratings = y[train_indices]

val_user_features = X_user[val_indices]
val_movie_features = X_movie[val_indices]
val_ratings = y[val_indices]

# Print feature dimensions
print("Feature dimensions:")
print(f"Movie features shape: {movie_features.shape}")
print(f"User features shape: {user_features.shape}")
print("\nTraining set shapes:")
print(f"User features: {train_user_features.shape}")
print(f"Movie features: {train_movie_features.shape}")
print(f"Ratings: {train_ratings.shape}")
```

Key changes and additions in this version:

1. **New Tag Feature Extraction**:

- Added `prepare_tag_features` function that:
  - Groups tags by movieId
  - Uses TF-IDF vectorization (from sklearn) to create tag features
  - Handles missing tags
  - Creates a feature vector of size `max_features` (default 1000)

2. **Enhanced Movie Features**:

- Now includes tag-based features along with:
  - Genre features (one-hot encoded)
  - Year features (extracted from title)
  - Title features (tokenized)
  - Tag features (TF-IDF vectors)

3. **Enhanced User Features**:

- Added tag activity metrics:
  - Number of tags created by user
  - Combined with existing rating statistics
  - All features normalized using StandardScaler

Benefits of Using TF-IDF for Tags:

1. **Importance Weighting**: TF-IDF automatically weights tags based on their frequency and importance
2. **Dimensionality Reduction**: Limits the feature space to most important tags
3. **Handling Sparse Data**: Effectively handles missing tags
4. **Text Preprocessing**: Automatically handles:
   - Stop word removal
   - Lowercase conversion
   - N-gram generation (1-2 words)

This enhanced version provides richer feature representations for both movies and users, which should lead to better recommendations. The tag features can capture nuanced aspects of movies that might not be present in just the genres or titles.

Would you like me to explain any specific part in more detail or show you how to build the neural network model that uses these features?


I'll show you how the processed data would look in tabular form at different stages. Let's break it down:

```python
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Load sample data (first few rows)
ratings_df = pd.read_csv('ratings.csv')
movies_df = pd.read_csv('movies.csv')
tags_df = pd.read_csv('tags.csv')

# 1. Original Data Format
print("1. Original Ratings Data:")
print(ratings_df.head())
print("\n2. Original Movies Data:")
print(movies_df.head())
print("\n3. Original Tags Data:")
print(tags_df.head())

# Process and show transformed data
def show_processed_data():
    # Process movie features
    genres = movies_df['genres'].str.get_dummies('|')

    # Process tags
    movie_tags = tags_df.groupby('movieId')['tag'].agg(lambda x: ' '.join(x)).reset_index()
    tfidf = TfidfVectorizer(max_features=5)  # Using 5 features for demonstration
    tag_features = tfidf.fit_transform(movie_tags['tag'].fillna('')).toarray()

    # Create sample processed data
    processed_data = pd.DataFrame({
        'movieId': ratings_df['movieId'].iloc[:5],
        'userId': ratings_df['userId'].iloc[:5],
        'rating': ratings_df['rating'].iloc[:5]
    })

    # Add movie features
    movie_features_df = pd.DataFrame(
        tag_features[:5],
        columns=[f'tag_feature_{i}' for i in range(5)]
    )

    # Add user features
    user_stats = ratings_df.groupby('userId').agg({
        'rating': ['mean', 'std', 'count']
    }).reset_index()
    user_stats.columns = ['userId', 'avg_rating', 'std_rating', 'rating_count']

    print("\n4. Processed Movie Features (Sample):")
    sample_movie_features = pd.concat([
        movies_df[['movieId', 'title']].head(),
        genres.head(),
        movie_features_df
    ], axis=1)
    print(sample_movie_features)

    print("\n5. Processed User Features (Sample):")
    print(user_stats.head())

    print("\n6. Final Training Data Format:")
    # Combine all features
    final_data = pd.DataFrame({
        'userId': ratings_df['userId'].iloc[:5],
        'movieId': ratings_df['movieId'].iloc[:5],
        'rating': ratings_df['rating'].iloc[:5],
    })
    final_data = final_data.merge(user_stats, on='userId', how='left')
    final_data = final_data.merge(sample_movie_features, on='movieId', how='left')
    print(final_data)

show_processed_data()
```

This will show the data at different stages. Let me explain each table's structure:

1. **Original Ratings Data**:

```
   userId  movieId  rating  timestamp
0      1     1193    4.0  978300760
1      1      661    3.0  978302109
2      1      914    3.0  978301968
3      1     3408    4.0  978300275
4      1     2355    5.0  978824291
```

2. **Original Movies Data**:

```
   movieId                               title                                        genres
0       1                    Toy Story (1995)   Adventure|Animation|Children|Comedy|Fantasy
1       2                      Jumanji (1995)               Adventure|Children|Fantasy
2       3             Grumpier Old Men (1995)                       Comedy|Romance
3       4            Waiting to Exhale (1995)                 Comedy|Drama|Romance
4       5  Father of the Bride Part II (1995)                           Comedy
```

3. **Original Tags Data**:

```
   userId  movieId  tag                   timestamp
0      15     1955  Great Acting          1445714994
1      15     2019  Visually Appealing    1445715207
2      15       17  Classic               1445715973
3      15      841  Stunning Visuals      1445716025
4      15     1891  Great Soundtrack      1445716039
```

4. **Processed Movie Features**:

```
   movieId  title    Action  Adventure  Comedy  Drama  tag_feature_1  tag_feature_2  tag_feature_3
0       1  ...         0         1       1      0        0.45         0.12          0.23
1       2  ...         0         1       0      0        0.34         0.56          0.11
2       3  ...         0         0       1      0        0.67         0.23          0.45
```

5. **Processed User Features**:

```
   userId  avg_rating  std_rating  rating_count  tag_count
0       1        3.8         0.9           48          5
1       2        4.2         0.7           98          12
2       3        3.5         1.1           75          3
```

6. **Final Training Data Format** (X_user, X_movie, y):

```
# X_user features:
[
    [3.8, 0.9, 48, 5],    # User 1 features: [avg_rating, std_rating, rating_count, tag_count]
    [4.2, 0.7, 98, 12],   # User 2 features
    ...
]

# X_movie features:
[
    [0,1,1,0, 0.45,0.12,0.23, ...],  # Movie 1 features: [genres..., tag_features..., title_features...]
    [0,1,0,0, 0.34,0.56,0.11, ...],  # Movie 2 features
    ...
]

# y (ratings):
[4.0, 3.0, 5.0, ...]  # Actual ratings
```

This final format is what goes into the neural network:

- X_user: Matrix of user features
- X_movie: Matrix of movie features
- y: Vector of ratings

Each training example consists of:

1. A user feature vector (user tower input)
2. A movie feature vector (movie tower input)
3. The corresponding rating (target value)

The neural network learns to predict the rating based on these user and movie feature vectors. The two towers process their respective features independently before combining them to make the final rating prediction.
